# Tentativo 19 · 3. Lunghezze ed embedding

**Ogni modello legge ogni articolo intero.** Si contano i token di ogni articolo con il tokenizer di ogni modello: il
limite (`MAX_TOKEN` = 4.096, più l'istruzione per Qwen) deve essere sopra l'articolo più lungo, cioè **nessun articolo
troncato**. ModernBERT arriva a 8.192 token, Qwen3-Embedding a 32.768, Mistral-Small a 32.768.

Poi i vettori dei due modelli congelati, sugli articoli anonimi, come nel tentativo 16: **Qwen3-Embedding-8B** (ultimo
token, con l'istruzione sull'affidabilità) e **Mistral-Small-24B Base** in 4 bit (media dei token, strati 27 e 40 di 40;
i modelli lineari usano il 27, il migliore nel tentativo 16).

In [1]:
MODELLO_BERT = "answerdotai/ModernBERT-base"
MODELLO_QWEN = "Qwen/Qwen3-Embedding-8B"
MODELLO_MISTRAL = "mistralai/Mistral-Small-24B-Base-2501"
STRATI = "27,40"
BUDGET_QWEN = 32768                 # token per lotto (lunghezza massima × articoli)
BUDGET_MISTRAL = 8192                # come il tentativo 16 (4 × 2.048): tiene in memoria tutti gli strati
GB_GPU = 22

In [2]:
import time
import warnings

import numpy as np
import pandas as pd

import modelli as Mo

warnings.filterwarnings("ignore")
OUT = Mo.Path("Results"); OUT.mkdir(exist_ok=True)
Mo.EMBEDDING.mkdir(parents=True, exist_ok=True)
art = pd.read_csv(Mo.DATASET / "articoli.csv.gz")
testi = art.text.fillna("").tolist()
def registra(msg, file=OUT / "avanzamento.log"):
    riga = f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {msg}"
    print(riga, flush=True)
    with open(file, "a") as f:
        f.write(riga + "\n")
print(f"{len(art):,} articoli"); print(Mo.ambiente())

25,524 articoli


{'python': '3.12.3', 'seme': 42, 'seme dello split': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2'}


## 1. Quanti token ha ogni articolo

In [3]:
righe = []
for nome, modello, prefisso in [("ModernBERT", MODELLO_BERT, ""), ("Qwen3 8B", MODELLO_QWEN, f"Instruct: {Mo.ISTRUZIONE_QWEN}\nQuery:"),
                                ("Mistral 24B", MODELLO_MISTRAL, "")]:
    n = Mo.conta_token(testi, modello, prefisso)
    limite = Mo.MAX_TOKEN + (64 if prefisso else 0)
    righe.append({"modello": nome, "mediana": int(np.median(n)), "99%": int(np.percentile(n, 99)), "massimo": int(n.max()),
                  "limite": limite, "articoli troncati": int((n > limite).sum())})
lung = pd.DataFrame(righe).set_index("modello"); lung.to_csv(OUT / "lunghezze_token.csv")
assert lung["articoli troncati"].sum() == 0, "qualche articolo supera il limite: alzare MAX_TOKEN"
lung

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


,mediana,99%,massimo,limite,articoli troncati
modello,,,,,
ModernBERT,764,1924,2635,4096,0
Qwen3 8B,759,1898,3000,4160,0
Mistral 24B,753,1912,2991,4096,0


## 2. Qwen3-Embedding-8B

In [4]:
file_qwen = Mo.EMBEDDING / "Qwen3-Embedding-8B.npy"
if not file_qwen.exists():
    Mo.aspetta_gpu(GB_GPU, registra)
    registra("Qwen3-Embedding-8B: inizio")
    E = Mo.embedding_qwen(testi, MODELLO_QWEN, BUDGET_QWEN, registra)
    np.save(file_qwen, E); registra(f"Qwen3-Embedding-8B: fatto, {E.shape}")
E = np.load(file_qwen); print(E.shape, "valori finiti:", np.isfinite(E).all())

2026-10-30 17:33:14  Qwen3-Embedding-8B: inizio


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:  25%|██▌       | 1/4 [00:01<00:05,  1.83s/it]

Loading checkpoint shards:  50%|█████     | 2/4 [00:03<00:03,  1.81s/it]

Loading checkpoint shards:  75%|███████▌  | 3/4 [00:05<00:01,  1.82s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.16s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.40s/it]

2026-10-30 17:33:41  Qwen: lotto 0, 1 s


2026-10-30 17:36:57  Qwen: lotto 200, 197 s


## 3. Mistral-Small-24B Base

In [ ]:
strati = [int(s) for s in str(STRATI).split(",")]
file_mistral = {s: Mo.EMBEDDING / f"Mistral-Small-24B_strato{s}.npy" for s in strati}
if not all(f.exists() for f in file_mistral.values()):
    Mo.aspetta_gpu(GB_GPU, registra)
    registra("Mistral-Small-24B: inizio")
    E = Mo.embedding_mistral(testi, MODELLO_MISTRAL, strati, BUDGET_MISTRAL, registra, Mo.EMBEDDING / "mistral_parziale.npy")
    for s in strati:
        np.save(file_mistral[s], E[s])
    (Mo.EMBEDDING / "mistral_parziale.npy").unlink(missing_ok=True)
    registra("Mistral-Small-24B: fatto")
for s, f in file_mistral.items():
    E = np.load(f); print(f"strato {s}: {E.shape}, valori finiti: {np.isfinite(E).all()}")
pd.Series(art.article_id).to_csv(Mo.EMBEDDING / "ordine_articoli.csv", index=False)     # le righe dei vettori